# Phase 6 -- Final Polish and Submission
## Human vs. AI-Generated Text Classification

**This is the last notebook in the pipeline.** It brings everything together:

1. **Load all results** from Phase 1--5 (scores, predictions, configs)
2. **Score dashboard** -- visualize the full progression from baseline to final ensemble
3. **Agreement analysis** -- how similar are the different submissions?
4. **Select top 2 submissions** for final selection
5. **Generate clean final submissions** in the exact required format
6. **Save a reproducibility snapshot** (all configs, scores, package versions)

**Submission format:**
- Metric: **Macro F1 Score**
- Format: (integer 0--5)


In [ ]:
import os, json, warnings, glob, time
os.chdir('/Users/aliivaezii/Documents/MALTO')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, confusion_matrix
from datetime import datetime

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
y_all = train_df['LABEL'].values

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Labels: {dict(zip(*np.unique(y_all, return_counts=True)))}')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## 1. Inventory All Submission Files

Let's see every CSV we've generated across all phases. For each file, we check:
- Header format
- Number of rows
- Label distribution

In [ ]:
# Scan all submission files
sub_files = sorted(glob.glob('submissions/*.csv'))

inventory = []
for path in sub_files:
    fname = os.path.basename(path)
    with open(path) as f:
        lines = f.readlines()
        header = lines[0].strip()
        n_rows = len(lines) - 1 # exclude header
 
        # Parse predictions
        preds = []
        for line in lines[1:]:
            parts = line.strip().split(',')
            if len(parts) >= 2:
                preds.append(int(parts[-1]))
            elif len(parts) == 1 and parts[0].isdigit():
                preds.append(int(parts[0]))
 
                dist = dict(zip(*np.unique(preds, return_counts=True))) if preds else {}
                inventory.append({
                    'file': fname,
                    'header': header,
                    'rows': n_rows,
                    'preds': np.array(preds),
                    'dist': dist,
                    })

print(f'Total submission files: {len(inventory)}')
print(f'\n{"File":<45} {"Header":<20} {"Rows":>5}')
print('-' * 75)
for item in inventory:
    print(f'{item["file"]:<45} {item["header"]:<20} {item["rows"]:>5}')

## 2. Load Phase Configs & Build Score Timeline

Each phase saved a config JSON with scores. We load them all to build a **progression chart** showing how our Macro F1 improved across the pipeline.

In [ ]:
# Load all available config files
configs = {}
for config_file in sorted(glob.glob('models/*.json')):
    name = os.path.basename(config_file)
    with open(config_file) as f:
        configs[name] = json.load(f)
        print(f'Loaded: {name}')

        # Build score progression
        # These are our best validated scores from each phase
score_timeline = []

# Phase 1: EDA + Baseline (from notebook 1 -- known values)
score_timeline.append(('Phase 1: TF-IDF+SVC Baseline', 0.8935, 'CV 5-fold'))
score_timeline.append(('Phase 1: TF-IDF+LR Baseline', 0.9305, 'Val split'))

# Phase 2: Feature Engineering (from classical_cv_results.json)
if 'classical_cv_results.json' in configs:
    cv_results = configs['classical_cv_results.json']
    best_classical = max(cv_results.values(), key=lambda x: x['mean_f1'])
    score_timeline.append((f'Phase 2: {best_classical["name"]}', best_classical['mean_f1'], 'CV 5-fold'))

    # Phase 3: Transformer (DeBERTa val F1)
splits = np.load('models/transformer_split_indices.npz')
val_idx = splits['val_idx']
val_preds = np.load('models/transformer_val_preds.npy')
deberta_val_f1 = f1_score(y_all[val_idx], val_preds, average='macro')
score_timeline.append(('Phase 3: DeBERTa-v3-base', deberta_val_f1, 'Val split'))

# Phase 4: Basic Ensemble (from ensemble_config.json)
if 'ensemble_config.json' in configs:
    d4 = configs['ensemble_config.json']
    w = d4['ensemble_weights']
    score_timeline.append((f'Phase 4: Soft Vote (D={w["deberta"]:.0%}/S={w["svc"]:.0%}/L={w["lr"]:.0%})', None, 'Cal set'))

    # Phase 5: Advanced Ensemble (from stacking_config.json)
if 'stacking_config.json' in configs:
    d5 = configs['stacking_config.json']
    for method_name, f1_val in d5.get('all_methods_ranked', []):
        score_timeline.append((f'Phase 5: {method_name}', f1_val, 'OOF'))

        # Display
print('\n' + '=' * 70)
print('SCORE PROGRESSION ACROSS ALL PHASES')
print('=' * 70)
print(f'{"Method":<55} {"F1":>8} {"Eval":>10}')
print('-' * 75)
for name, f1, eval_type in score_timeline:
    f1_str = f'{f1:.4f}' if f1 is not None else ' --'
    print(f'{name:<55} {f1_str:>8} {eval_type:>10}')

### 📈 Score Progression Across Phases

Load model configs from each phase to reconstruct the full score progression from baseline to final ensemble.

In [ ]:
# Score progression bar chart
# Filter to entries with actual scores
scored = [(name, f1) for name, f1, _ in score_timeline if f1 is not None]

fig, ax = plt.subplots(figsize=(14, max(6, len(scored) * 0.5)))

names = [s[0] for s in scored]
scores = [s[1] for s in scored]

# Color by phase
phase_colors = {
    'Phase 1': '#95a5a6', # gray
    'Phase 2': '#3498db', # blue
    'Phase 3': '#e74c3c', # red
    'Phase 4': '#f39c12', # orange
    'Phase 5': '#2ecc71', # green
    }
colors = []
for name in names:
    phase = name.split(':')[0]
    colors.append(phase_colors.get(phase, '#7f8c8d'))

bars = ax.barh(range(len(names)), scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Macro F1 Score', fontsize=12)
ax.set_title('Score Progression Across Phases', fontweight='bold', fontsize=14)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
        f'{score:.4f}', va='center', fontweight='bold', fontsize=9)

    # Zoom into the relevant range
ax.set_xlim(min(scores) - 0.02, max(scores) + 0.02)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/score_progression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/score_progression.png')

## 3. Submission Agreement Analysis

Before selecting our final 2 submissions, let's see how similar the different predictions are. If two methods give nearly identical predictions, there's no point submitting both.

We compute a **pairwise agreement matrix** -- what percentage of test samples get the same label.

In [ ]:
# Filter to valid submission files (600 predictions, correct header)
valid_subs = [item for item in inventory if item['rows'] == 600 and len(item['preds']) == 600]
print(f'Valid submission files: {len(valid_subs)}')

# Pairwise agreement matrix
n = len(valid_subs)
agreement = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        agreement[i, j] = (valid_subs[i]['preds'] == valid_subs[j]['preds']).mean() * 100

names = [item['file'].replace('.csv', '') for item in valid_subs]

fig, ax = plt.subplots(figsize=(max(10, n * 0.8), max(8, n * 0.6)))
sns.heatmap(agreement, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax,
    xticklabels=names, yticklabels=names,
    vmin=80, vmax=100)
ax.set_title('Submission Pairwise Agreement (%)', fontweight='bold', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('figures/submission_agreement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/submission_agreement.png')

## 4. Label Distribution Comparison

Do different methods predict drastically different label distributions? This can reveal over/under-prediction of minority classes.

In [ ]:
# Collect label distributions from valid submissions
dist_data = []
for item in valid_subs:
    for label in range(NUM_LABELS):
        count = (item['preds'] == label).sum()
        dist_data.append({
            'submission': item['file'].replace('.csv', ''),
            'class': LABEL_MAP[label],
            'count': count,
            })

dist_df = pd.DataFrame(dist_data)

# Grouped bar chart -- train distribution as reference
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: Train distribution for reference
train_dist = pd.Series(y_all).value_counts().sort_index()
train_pct = train_dist / train_dist.sum() * 100
axes[0].bar([LABEL_MAP[i] for i in range(NUM_LABELS)], train_pct.values,
    color=['#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#3498db', '#1abc9c'],
    edgecolor='black', linewidth=0.5)
axes[0].set_title('Training Set Distribution (%)', fontweight='bold')
axes[0].set_ylabel('Percentage')
for i, pct in enumerate(train_pct.values):
    axes[0].text(i, pct + 0.5, f'{pct:.1f}%', ha='center', fontsize=9)

    # Right: Predicted distributions for selected submissions
    # Pick a subset of submissions for readability
selected = [item for item in valid_subs if 'phase5' in item['file'] or 'deberta' in item['file']]
if len(selected) == 0:
    selected = valid_subs[:min(5, len(valid_subs))]

x = np.arange(NUM_LABELS)
width = 0.8 / max(len(selected), 1)
for idx, item in enumerate(selected):
    counts = [(item['preds'] == label).sum() for label in range(NUM_LABELS)]
    pcts = [c / 600 * 100 for c in counts]
    axes[1].bar(x + idx * width, pcts, width, label=item['file'].replace('.csv', ''),
        edgecolor='black', linewidth=0.3)

axes[1].set_xticks(x + width * len(selected) / 2)
axes[1].set_xticklabels([LABEL_MAP[i] for i in range(NUM_LABELS)], rotation=30)
axes[1].set_title('Predicted Distributions (%)', fontweight='bold')
axes[1].set_ylabel('Percentage')
axes[1].legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig('figures/label_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/label_distribution_comparison.png')

## 5. Select Top 2 Final Submissions

**Strategy for selecting 2 final submissions:**

1. **Submission A (Safe):** The method with the highest OOF/CV F1 -- our most reliable prediction
2. **Submission B (Diverse):** A method that disagrees with Submission A on enough samples to provide diversity -- in case our validation estimate is off

The idea: if Submission A is overfit to our validation split, Submission B might generalize better because it uses a different approach.

In [ ]:
# Load Phase 5 config to know the best method
if 'stacking_config.json' in configs:
    p5 = configs['stacking_config.json']
    ranked_methods = p5.get('all_methods_ranked', [])
    print('Phase 5 ranked methods:')
    for i, (name, f1) in enumerate(ranked_methods):
        print(f' {i+1}. {name:<35} F1={f1:.4f}')
else:
    ranked_methods = []
    print('Phase 5 config not found -- using available submissions.')

    # Map Phase 5 methods to submission files
method_to_file = {
    'Phase 4: Soft Voting': 'phase4_softvote',
    'Stacking (Meta-LR)': 'phase5_stacking',
    'Rank Averaging': 'phase5_rank_avg',
    'Specialist Voting': 'phase5_specialist',
    'Confidence Blending': 'phase5_conf_blend',
    'Stacking + Thresholds': 'phase5_stack_thresh',
    }

# Submission A: highest F1
sub_a_name = None
sub_a_preds = None
for method_name, f1 in ranked_methods:
    short = method_to_file.get(method_name, '')
    matches = [item for item in valid_subs if short in item['file']]
    if matches:
        sub_a_name = matches[0]['file']
        sub_a_preds = matches[0]['preds']
        sub_a_f1 = f1
        break

        # Fallback: use the first valid submission if Phase 5 config is unavailable
if sub_a_name is None and valid_subs:
    sub_a_name = valid_subs[0]['file']
    sub_a_preds = valid_subs[0]['preds']
    sub_a_f1 = 0.0

print(f'\n→ Submission A (Safe): {sub_a_name} (F1={sub_a_f1:.4f})')

# Submission B: most disagreement with A among high-scoring methods
best_disagree = 0
sub_b_name = None
sub_b_preds = None
sub_b_f1 = 0.0

for item in valid_subs:
    if item['file'] == sub_a_name:
        continue
        if len(item['preds']) != 600:
            continue
            disagree = (item['preds'] != sub_a_preds).sum()
            if disagree > best_disagree:
                best_disagree = disagree
                sub_b_name = item['file']
                sub_b_preds = item['preds']

print(f'→ Submission B (Diverse): {sub_b_name} ({best_disagree} samples differ from A)')

# Show what differs
if sub_a_preds is not None and sub_b_preds is not None:
    agree_pct = (sub_a_preds == sub_b_preds).mean() * 100
    print(f'\nAgreement: {agree_pct:.1f}% ({(sub_a_preds == sub_b_preds).sum()}/600)')
 
    # Where they disagree, what classes are involved?
    diff_mask = sub_a_preds != sub_b_preds
    print(f'\nDisagreements by class (Submission A → B):')
    for cls in range(NUM_LABELS):
        a_count = ((sub_a_preds == cls) & diff_mask).sum()
        b_count = ((sub_b_preds == cls) & diff_mask).sum()
        print(f' {LABEL_MAP[cls]:10s}: A predicts {a_count}, B predicts {b_count}')

## 6. Generate Clean Final Submissions

Create the two final submission files with the exact format required:
- Header: `ID,LABEL`
- 600 rows: `0,3` (ID is 0-indexed integer, LABEL is 0--5)
- No extra whitespace, no BOM, Unix line endings

In [ ]:
def write_clean_submission(preds, path):
    """Write a submission CSV in the exact required format."""
    lines = ['ID,LABEL'] + [f'{i},{int(preds[i])}' for i in range(600)]
    with open(path, 'w', newline='\n') as f:
        f.write('\n'.join(lines) + '\n')
 
        # Verify
        with open(path) as f:
            content = f.readlines()
            assert content[0].strip() == 'ID,LABEL', f'Bad header: {content[0]}'
            assert len(content) == 601, f'Bad row count: {len(content)}'
            return path


            # Write final submissions
os.makedirs('submissions', exist_ok=True)

path_a = write_clean_submission(sub_a_preds, 'submissions/final_submission_A.csv')
path_b = write_clean_submission(sub_b_preds, 'submissions/final_submission_B.csv')

print(f' Final Submission A: {path_a}')
print(f' Source: {sub_a_name}')
print(f' Distribution: {dict(zip(*np.unique(sub_a_preds, return_counts=True)))}')

print(f'\n Final Submission B: {path_b}')
print(f' Source: {sub_b_name}')
print(f' Distribution: {dict(zip(*np.unique(sub_b_preds, return_counts=True)))}')

# Preview first 5 lines
print('\n--- Preview (Submission A) ---')
with open(path_a) as f:
    for i, line in enumerate(f):
        if i < 6:
            print(f' {line.strip()}')
        else:
            break
print(' ...')

## 7. Final Confusion Matrices (OOF)

Show the confusion matrix for our two selected approaches side-by-side. This uses OOF/validation predictions (the only labeled data we have).

In [ ]:
# Load OOF data for selected methods
# We'll use the Phase 5 stacking OOF (most likely best) and Phase 3 DeBERTa val
splits = np.load('models/transformer_split_indices.npz')
val_idx = splits['val_idx']
cal_idx = splits['cal_idx']

deberta_val_preds = np.load('models/transformer_val_preds.npy')
deberta_val_f1 = f1_score(y_all[val_idx], deberta_val_preds, average='macro')

# SVC OOF (full 5-fold)
svc_oof = np.load('models/ensemble_svc_oof_probs.npy')
svc_oof_preds = svc_oof.argmax(axis=1)
svc_oof_f1 = f1_score(y_all, svc_oof_preds, average='macro')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DeBERTa validation confusion
cm1 = confusion_matrix(y_all[val_idx], deberta_val_preds)
cm1_pct = cm1.astype(float) / cm1.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm1_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[0],
    xticklabels=[LABEL_MAP[i] for i in range(6)],
    yticklabels=[LABEL_MAP[i] for i in range(6)])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'DeBERTa-v3-base (Val)\nMacro F1 = {deberta_val_f1:.4f}', fontweight='bold')

# SVC 5-fold OOF confusion
cm2 = confusion_matrix(y_all, svc_oof_preds)
cm2_pct = cm2.astype(float) / cm2.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm2_pct, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1],
    xticklabels=[LABEL_MAP[i] for i in range(6)],
    yticklabels=[LABEL_MAP[i] for i in range(6)])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title(f'SVC (C=5.0) 5-Fold OOF\nMacro F1 = {svc_oof_f1:.4f}', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/final_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/final_confusion_matrices.png')

## 8. Reproducibility Snapshot

Save a complete record of the pipeline for reproducibility:
- All config files merged
- Package versions
- File checksums
- Score summary

In [ ]:
import hashlib
import sys

def file_md5(path):
    """Compute MD5 hash of a file."""
    h = hashlib.md5()
    with open(path, 'rb') as f:
        h.update(f.read())
        return h.hexdigest()


snapshot = {
    'timestamp': datetime.now().isoformat(),
    'python_version': sys.version,
    'packages': {},
    'score_timeline': [(name, float(f1) if f1 else None, eval_type) for name, f1, eval_type in score_timeline],
    'final_submission_a': {
    'file': 'submissions/final_submission_A.csv',
    'source': sub_a_name,
    'md5': file_md5('submissions/final_submission_A.csv'),
    'distribution': {LABEL_MAP[k]: int(v) for k, v in zip(*np.unique(sub_a_preds, return_counts=True))},
    },
    'final_submission_b': {
    'file': 'submissions/final_submission_B.csv',
    'source': sub_b_name,
    'md5': file_md5('submissions/final_submission_B.csv'),
    'distribution': {LABEL_MAP[k]: int(v) for k, v in zip(*np.unique(sub_b_preds, return_counts=True))},
    },
    'all_configs': configs,
    }

# Collect package versions
for pkg in ['numpy', 'pandas', 'sklearn', 'torch', 'transformers', 'xgboost', 'lightgbm', 'joblib']:
    module = sys.modules.get(pkg) or __import__(pkg, fromlist=['__version__'])
    version = getattr(module, '__version__', 'unknown')
    snapshot['packages'][pkg] = version

with open('models/final_snapshot.json', 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)

print('Reproducibility snapshot saved: models/final_snapshot.json')
print(f'\nPackage versions:')
for pkg, ver in snapshot['packages'].items():
    print(f' {pkg:15s} {ver}')

## 9. Submit to Kaggle (Optional)

Run these cells to automatically submit the final files to Kaggle. You can also submit manually through the Kaggle website.

In [ ]:
# Set up Kaggle credentials
os.environ['PATH'] += ':/Users/aliivaezii/Library/Python/3.9/bin'
os.environ['KAGGLE_USERNAME'] = 'aliivaezii'
os.environ['KAGGLE_KEY'] = 'KGAT_f881721a6d4a99d83c1fe701d4aa6f64'

COMPETITION = 'malto-recruitment-hackathon'

# Submit A
print('Submitting Final A...')
cmd_a = f'kaggle competitions submit -c {COMPETITION} -f submissions/final_submission_A.csv -m "Phase6 Final A: best ensemble method"'
result_a = os.popen(cmd_a).read()
print(result_a)

# Submit B
print('Submitting Final B...')
cmd_b = f'kaggle competitions submit -c {COMPETITION} -f submissions/final_submission_B.csv -m "Phase6 Final B: diverse ensemble method"'
result_b = os.popen(cmd_b).read()
print(result_b)

print('\n Both submissions sent!')
print('Check leaderboard: https://www.kaggle.com/competitions/malto-recruitment-hackathon/leaderboard')

## 10. Project Summary Dashboard

A final summary of the entire project -- what we did, what worked, and what the key results are.

In [ ]:
print('=' * 70)
print('FINAL PROJECT SUMMARY')
print('=' * 70)

print('\n PIPELINE OVERVIEW:')
print(' Phase 1: EDA + Baseline (TF-IDF + LR/SVC)')
print(' Phase 2: Feature Engineering + 10 Classical Models')
print(' Phase 3: DeBERTa-v3-base Fine-Tuning')
print(' Phase 4: Basic Ensemble (Soft Voting + Pseudo-Labeling)')
print(' Phase 5: Advanced Ensemble (Stacking + Rank Avg + Specialist)')
print(' Phase 6: Final Polish & Submission (this notebook)')

print('\n KEY RESULTS:')
for name, f1, eval_type in score_timeline:
    if f1 is not None:
        print(f' {name:<55} F1={f1:.4f} ({eval_type})')

print('\n FINAL SUBMISSIONS:')
print(f' A: {sub_a_name}')
print(f' Distribution: {dict(zip(*np.unique(sub_a_preds, return_counts=True)))}')
print(f' B: {sub_b_name}')
print(f' Distribution: {dict(zip(*np.unique(sub_b_preds, return_counts=True)))}')

# Count artifacts
n_models = len(glob.glob('models/*'))
n_subs = len(glob.glob('submissions/*.csv'))
n_figs = len(glob.glob('figures/*.png'))
n_notebooks = len(glob.glob('notebooks/*.ipynb'))

print(f'\n ARTIFACTS:')
print(f' Notebooks: {n_notebooks}')
print(f' Models/data: {n_models} files in models/')
print(f' Submissions: {n_subs} CSV files')
print(f' Figures: {n_figs} PNG files')

print('\n KEY INSIGHTS:')
print(' 1. Misspelling ratio is the strongest single feature (Human >> AI)')
print(' 2. Class imbalance (19:1) requires balanced weights everywhere')
print(' 3. DeBERTa-v3-base with Focal Loss handles minority classes well')
print(' 4. Ensemble of DeBERTa + SVC + LR outperforms any single model')
print(' 5. Per-class threshold optimization boosts minority class recall')

print('\n Project complete!')
print('\n Repository: https://github.com/aliivaezii/MALTO')